# Building a multi-user ReAct AI Agent Chatbot with LangChain - Legacy Syntax

This demo will cover building AI Agents with the legacy LangChain `AgentExecutor`. These are fine for getting started, but for working with more advanced agents, LangChain recommends to use LangGraph, which we will cover in the future demos.

Agents are systems that use an LLM as a reasoning engine to determine which actions to take and what the inputs to those actions should be. The results of those actions can then be fed back into the agent and it determines whether more actions are needed, or whether it is okay to stop.



## Install OpenAI, and LangChain dependencies

In [ ]:
!pip install langchain==0.3.11
!pip install langchain-openai==0.2.12
!pip install langchain-community==0.3.11

## Enter Open AI API Key

In [ ]:
from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

## Enter Tavily Search API Key

Get a free API key from [here](https://tavily.com/#api)

In [ ]:
TAVILY_API_KEY = getpass('Enter Tavily Search API Key: ')

Enter Tavily Search API Key: ··········


## Enter WeatherAPI API Key

Get a free API key from [here](https://www.weatherapi.com/signup.aspx)

In [ ]:
WEATHER_API_KEY = getpass('Enter WeatherAPI API Key: ')

## Setup Environment Variables

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

## Create Tools

Here we create two custom tools which are wrappers on top of the [Tavily API](https://tavily.com/#api) and [WeatherAPI](https://www.weatherapi.com/)

- Simple Web Search tool
- Weather tool

In [7]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool
import requests
import json

tv_search = TavilySearchResults(max_results=3, search_depth='advanced',
                                max_tokens=10000)

@tool
def search_web(query: str) -> list:
    """Search the web for a query."""
    tavily_tool = TavilySearchResults(max_results=2)
    results = tavily_tool.invoke(query)
    return results

@tool
def get_weather(query: str) -> list:
    """Search weatherapi to get the current weather."""
    base_url = "http://api.weatherapi.com/v1/current.json"
    complete_url = f"{base_url}?key={WEATHER_API_KEY}&q={query}"

    response = requests.get(complete_url)
    data = response.json()
    if data.get("location"):
        return data
    else:
        return "Weather Data Not Found"

## Test Tool Calling with LLM

In [8]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [search_web, get_weather]

chatgpt_with_tools = chatgpt.bind_tools(tools)

In [9]:
prompt = "who won the champions league in 2024"
response = chatgpt_with_tools.invoke(prompt)
response.tool_calls

[{'name': 'search_web',
  'args': {'query': 'Champions League 2024 winner'},
  'id': 'call_91IZxTilalvwjCkEZqa9SdQZ',
  'type': 'tool_call'}]

In [10]:
prompt = "how is the weather in Bangalore today"
response = chatgpt_with_tools.invoke(prompt)
response.tool_calls

[{'name': 'get_weather',
  'args': {'query': 'Bangalore'},
  'id': 'call_8yd95TD0qiUerVjQTisJ8voH',
  'type': 'tool_call'}]

## Build and Test AI Agent

Now that we have defined the tools and the LLM, we can create the agent. We will be using a tool calling agent to bind the tools to the agent with a prompt. We will also add in the capability to store historical conversations as memory

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYS_PROMPT = """Act as a helpful assistant.
                Use the tools at your disposal to perform tasks as needed.
                  - get_weather: whenever user asks get the weather of a place.
                  - search_web: whenever user asks for information on current events or if you don't know the answer.
             """

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        MessagesPlaceholder(variable_name="history", optional=True),
        ("human", "{query}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

prompt_template.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="Act as a helpful assistant.\n                Use the tools at your disposal to perform tasks as needed.\n                  - get_weather: whenever user asks get the weather of a place.\n                  - search_web: whenever user asks for information on current events or if you don't know the answer.\n             "), additional_kwargs={}),
 MessagesPlaceholder(variable_name='history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

Now, we can initalize the agent with the LLM, the prompt, and the tools.

The agent is responsible for taking in input and deciding what actions to take.

REMEMBER the Agent does not execute those actions - that is done by the AgentExecutor

Note that we are passing in the model `chatgpt`, not `chatgpt_with_tools`.

That is because `create_tool_calling_agent` will call `.bind_tools` for us under the hood.

This should ideally be used with an LLM which supports tool \ function calling

In [12]:
from langchain.agents import create_tool_calling_agent

agent = create_tool_calling_agent(chatgpt, tools, prompt_template)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'query'], optional_variables=['history'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk,

Finally, we combine the `agent` (the brains) with the `tools` inside the `AgentExecutor` (which will repeatedly call the agent and execute tools).

In [13]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools)
agent_executor

AgentExecutor(verbose=False, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'query'], optional_variables=['history'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChun

In [14]:
query = """Tell me who won the champions league in 2024,
            show some detailed information about the match also
        """
response = chatgpt.invoke(query)
response.content

"I'm sorry, but I don't have access to real-time data or events beyond October 2023. To find out who won the UEFA Champions League in 2024 and get detailed information about the match, I recommend checking the latest sports news websites, the official UEFA website, or other reliable sources that provide updates on football events."

In [15]:
query = """Tell me who won the champions league in 2024,
            show some detailed information about the match also
        """
response = agent_executor.invoke({"query": query})

In [16]:
response

{'query': 'Tell me who won the champions league in 2024,\n            show some detailed information about the match also\n        ',
 'output': 'The 2024 UEFA Champions League final was held at Wembley Stadium in London, England, on June 1, 2024. The match was contested between German club Borussia Dortmund and Spanish club Real Madrid.\n\nReal Madrid emerged victorious in the final, defeating Borussia Dortmund. The match was challenging for Real Madrid during the first hour, but they secured the win with goals from Dani Carvajal and Vinicius Junior.\n\nFor more detailed information, you can visit the [Wikipedia page](https://en.wikipedia.org/wiki/2024_UEFA_Champions_League_Final) or check out the [Sporting News article](https://www.sportingnews.com/us/soccer/news/who-won-champions-league-final-2024-real-madrid-borussia-dortmund/c2aa2d0b8666ccb0125474be).'}

In [17]:
from IPython.display import display, Markdown

display(Markdown(response['output']))

The 2024 UEFA Champions League final was held at Wembley Stadium in London, England, on June 1, 2024. The match was contested between German club Borussia Dortmund and Spanish club Real Madrid.

Real Madrid emerged victorious in the final, defeating Borussia Dortmund. The match was challenging for Real Madrid during the first hour, but they secured the win with goals from Dani Carvajal and Vinicius Junior.

For more detailed information, you can visit the [Wikipedia page](https://en.wikipedia.org/wiki/2024_UEFA_Champions_League_Final) or check out the [Sporting News article](https://www.sportingnews.com/us/soccer/news/who-won-champions-league-final-2024-real-madrid-borussia-dortmund/c2aa2d0b8666ccb0125474be).

In [18]:
query = """how is the weather in Bangalore today?
        """
response = agent_executor.invoke({"query": query})
display(Markdown(response['output']))

The weather in Bangalore today is currently experiencing light rain. The temperature is 21.3°C (70.3°F), with a humidity level of 94%. The wind is blowing from the east at 20.5 kph (12.8 mph). Visibility is somewhat limited at 1.2 km.

In [19]:
query = """how is the weather in Dubai today?
        """
response = agent_executor.invoke({"query": query})
display(Markdown(response['output']))

The weather in Dubai today is sunny with a temperature of 20.4°C (68.7°F). The wind is blowing from the southeast at 9.7 kph (6.0 mph), and the humidity is at 53%. There is no precipitation, and the visibility is 10 km (6 miles). The UV index is currently at 0.

In [20]:
query = """which city is hotter?
        """
response = agent_executor.invoke({"query": query})
display(Markdown(response['output']))

Could you please specify the cities you would like to compare in terms of temperature?

The agent is doing pretty well but unfortunately it doesn't remember conversations. Let's now use some memory to store this.

## Build and Test Multi-User Conversational AI Agent

We will now use `SQLChatMessageHistory` which we learnt in the previous module to store separate conversation histories per user or session.

This will help us build a conversational Agentic Chatbot which will be accessed by many users at the same time

In [21]:
# removes the memory database file - usually not needed
# you can run this only when you want to remove ALL conversation histories
# ok if you get rm: cannot remove 'memory.db': No such file or directory  because initially no memory exists
!rm memory.db

rm: cannot remove 'memory.db': No such file or directory


In [22]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# used to retrieve conversation history from database
# based on a specific user or session ID
def get_session_history_db(session_id):
    return SQLChatMessageHistory(session_id, "sqlite:///memory.db")

# create a conversation chain + agent which can load memory based on specific user or session id
agentic_chatbot = RunnableWithMessageHistory(
    agent_executor,
    get_session_history_db,
    input_messages_key="query",
    history_messages_key="history",
)

# function to call the agent show results per user session
from IPython.display import display, Markdown
def chat_with_agent(prompt: str, session_id: str):
    response = agentic_chatbot.invoke({"query": prompt},
                                      {'configurable': { 'session_id': session_id}})
    display(Markdown(response['output']))

Let's now simulate User 1 using the agent

In [23]:
user_id = 'john001'
prompt = "Hi can you tell me who won champions league in 2024"
chat_with_agent(prompt, user_id)

/home/santhosh/anaconda3/lib/python3.11/site-packages/langchain_core/runnables/history.py:608: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  message_history = self.get_session_history(


Real Madrid won the 2024 UEFA Champions League, defeating Borussia Dortmund 2-0 in the final. This victory marked Real Madrid's 15th European title.

In [24]:
prompt = "Tell me more about this event in detail please"
chat_with_agent(prompt, user_id)

In the 2024 UEFA Champions League final, Real Madrid triumphed over Borussia Dortmund with a 2-0 victory. The match took place on June 1, 2024. Real Madrid secured their 15th European title with two goals in the final 20 minutes of the game. This victory further solidified Real Madrid's position as the club with the most European titles. You can find more details and highlights of the match on [USA Today](https://www.usatoday.com/story/sports/soccer/2024/06/01/real-madrid-dortmund-champions-league-final-live-score-highlights/73916662007/) and [ESPN](https://www.espn.com/soccer/match/_/gameId/702410/real-madrid-borussia-dortmund).

Let's now simulate User 2 using the agent

In [25]:
user_id = 'bond007'
prompt = "how is the weather in Bangalore today? Show detailed statistics"
chat_with_agent(prompt, user_id)

The current weather in Bangalore is as follows:

- **Condition**: Light rain
- **Temperature**: 21.3°C (70.3°F)
- **Feels Like**: 21.3°C (70.3°F)
- **Wind**: 12.8 mph (20.5 kph) from the East
- **Wind Gusts**: Up to 16.1 mph (25.9 kph)
- **Humidity**: 94%
- **Pressure**: 1018.0 mb (30.06 in)
- **Precipitation**: 0.02 mm
- **Cloud Cover**: 75%
- **Visibility**: 1.2 km
- **UV Index**: 2.4

The weather is updated as of 09:30 local time.

In [26]:
user_id = 'bond007'
prompt = "what about Dubai?"
chat_with_agent(prompt, user_id)

The current weather in Dubai is as follows:

- **Condition**: Sunny
- **Temperature**: 20.4°C (68.7°F)
- **Feels Like**: 20.4°C (68.7°F)
- **Wind**: 6.0 mph (9.7 kph) from the Southeast
- **Wind Gusts**: Up to 10.2 mph (16.4 kph)
- **Humidity**: 53%
- **Pressure**: 1017.0 mb (30.03 in)
- **Precipitation**: 0.0 mm
- **Cloud Cover**: 0%
- **Visibility**: 10.0 km
- **UV Index**: 0.0

The weather is updated as of 07:30 local time.

In [27]:
user_id = 'bond007'
prompt = "which city is hotter?"
chat_with_agent(prompt, user_id)

Currently, Bangalore is hotter than Dubai. The temperature in Bangalore is 21.3°C (70.3°F), while in Dubai, it is 20.4°C (68.7°F).